# Recorded Metrics Analysis

Analyze actual JSONL recordings from one local history. This notebook does not load the dummy dataset or apply its expected totals. Copy Android recordings to your analysis machine before opening them.

Select the recording folder below, then choose **Run All**. Calculations are shared with the test notebook and CSV exporter through `metrics_analysis.py`.


## 1. Configuration

The default Windows folder matches the current project's company and product names. Confirm the actual path with **MetricsRecorder &rarr; Print metrics file path**. For Android, copy the session files to a local folder and set `DATA_DIR` to that folder. Alternatively, set the `METRICS_DATA_DIR` environment variable before starting Jupyter.

Keep a separate folder for each recording history. Include all attempts from its beginning when interpreting attempts to boss. `GAME_VERSION` filters the notebook's views. Export remains opt-in.


In [ ]:
# Select actual recordings; never fall back to the dummy dataset.
# --- CONFIGURATION ---
from pathlib import Path
import sys
import tempfile
import os

candidates = [Path.cwd(), Path.cwd() / 'Tools/Metrics',
              Path.cwd() / 'LABAGI/Tools/Metrics']
TOOL_DIR = next((p for p in candidates if (p / 'metrics_analysis.py').exists()), None)
if TOOL_DIR is None:
    raise FileNotFoundError('Open the notebook from Tools/Metrics or the project root.')
sys.path.insert(0, str(TOOL_DIR))

default_data = Path.home() / 'AppData/LocalLow/DefaultCompany/EchoesBelow/Metrics'
DATA_DIR = Path(os.environ.get('METRICS_DATA_DIR', default_data)).expanduser()
if DATA_DIR.resolve() == (TOOL_DIR / 'test_data').resolve():
    raise ValueError('Use MetricsAnalysis.ipynb for the dummy dataset.')
if not DATA_DIR.is_dir() or not any(DATA_DIR.glob('*.jsonl')):
    raise FileNotFoundError(f'No recordings found in {DATA_DIR}. Set DATA_DIR to a folder containing session JSONL files.')
GAME_VERSION = None  # None includes all recorded versions
EXPORT = False
OUTPUT_DIR = Path(tempfile.gettempdir()) / 'MetricsAnalysis'


In [ ]:
# Shared formatting keeps the tables and charts consistent.
# --- PRESENTATION ---
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display, Markdown
# Reload after source changes when reusing an existing notebook kernel.
import importlib
import metrics_analysis
importlib.reload(metrics_analysis)
from metrics_analysis import analyze, load_records

plt.rcParams.update({'figure.figsize': (10, 4), 'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'bold', 'axes.labelcolor': '#334155',
    'text.color': '#0f172a', 'font.size': 10, 'figure.facecolor': 'white'})
BLUE, TEAL, AMBER = '#3563ad', '#167d8d', '#c58b32'

def table(frame):
    # Leave table styling to Jupyter's default renderer.
    display(frame)


def finish(ax, title, ylabel):
    ax.set_title(title, loc='left', pad=15)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=.15)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()


## 2. Data Quality

Malformed lines and duplicate session/sequence pairs are reported. A decision without an end is **incomplete**. A history without a boss arrival remains **not reached**. Missing events do not prove that a feature was used or that its measured result was zero.


In [ ]:
# Inspect coverage and warnings before interpreting the measurements.
records, read_warnings = load_records(DATA_DIR)
if GAME_VERSION is not None:
    records = [row for row in records if row['gameVersion'] == GAME_VERSION]

result = analyze(records, read_warnings)

display(Markdown('**Recorded Gameplay Data**'))
print('Input folder:', DATA_DIR)
print(f'{len(records)} records : {len({r["sessionId"] for r in records})} sessions')
print('Versions:', ', '.join(sorted({r['gameVersion'] for r in records})) or 'no data')

table(pd.DataFrame(sorted(result['coverage'].items()), columns=['Event', 'Records']))
if result['warnings']:
    print('Warnings to review:')
    for warning in result['warnings']:
        print(' -', warning)
else:
    print('No read or consistency warnings in the selected records.')
    
open_sessions = {r['sessionId'] for r in records if r['eventName'] == 'session.started'} - {
    r['sessionId'] for r in records if r['eventName'] == 'session.ended'}
print('Sessions without a recorded end:', len(open_sessions))


## 3. Room Exploration

`room.entered` records an entry; `room.completed` records the room's completion condition. Each room ID counts once per attempt, while the entry count includes revisits. Summing attempts measures total room exposure, not the number of different rooms in the map.


In [ ]:
# Count room IDs once per attempt; keep repeated entries separately.
rooms = pd.DataFrame(result['rooms'])
if rooms.empty:
    display(Markdown('No attempts with a recorded start.'))
else:
    table(rooms)
    print(f"Visited: {rooms.visited.sum()} : Completed: {rooms.completed.sum()} : Entries: {rooms.entries.sum()}")
    labels = rooms['runId']
    ax = rooms.set_axis(labels)[['visited', 'completed']].rename(
        columns={'visited': 'Visited', 'completed': 'Completed'}).plot.bar(color=[BLUE, TEAL], rot=0)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    finish(ax, 'Rooms per Attempt', 'Number of rooms')


## 4. Combat Progression

Each combat has a boolean victory result. Count those results once across the selected history to obtain cumulative victories by attempt. Damage events identify the combat as subject; missing measurements remain unavailable.


In [ ]:
# Count victories within each attempt, then compare successive attempts.
combats = pd.DataFrame(result['combats'])
progression = pd.DataFrame(result['progression'])
if progression.empty:
    display(Markdown('No recorded attempts.'))
else:
    table(progression)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for keys, group in progression.groupby(['gameVersion']):
        label = keys[0] if isinstance(keys, tuple) else str(keys)
        axes[0].plot(group.attempt, group.cumulativeVictories, marker='o', label=label)
        # Plot only measurements actually recorded for this attempt.
        for column, style in [('damageDealt', '-'), ('damageReceived', '--')]:
            available = group.dropna(subset=[column])
            if not available.empty:
                axes[1].plot(available.attempt, available[column], style, marker='o', label=label + ' / ' + column)
    for ax, title, ylabel in zip(axes, ['Cumulative Victories by Attempt', 'Damage per Attempt'], ['Victories', 'Damage']):
        ax.set_title(title, loc='left')
        ax.set_xlabel('Attempt')
        ax.set_ylabel(ylabel)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(axis='y', alpha=.15)
        if ax.lines:
            ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()


## 5. Decision Time

Contexts are configurable: `path`, `card`, `target`, `reward`, or another name. The recorder measures active time from making a choice available to confirming it, subtracting explicit pauses. Cancelled and incomplete decisions are shown separately and are excluded from confirmed-time averages and medians.


In [ ]:
# Only confirmed decisions contribute to mean and median durations.
decisions = pd.DataFrame(result['decisions'])
summary = pd.DataFrame(result['decision_summary'])
if summary.empty:
    display(Markdown('No recorded decisions.'))
else:
    table(summary)
    confirmed = decisions[decisions.outcome == 'confirmed']
    if not confirmed.empty:
        labels = summary.gameVersion + ' / ' + summary.context
        chart = summary.set_axis(labels)[['meanSeconds', 'medianSeconds']].dropna(how='all')
        ax = chart.rename(columns={'meanSeconds': 'Mean', 'medianSeconds': 'Median'}).plot.bar(
            color=[BLUE, AMBER], rot=15)
        finish(ax, 'Active Decision Time by Context', 'Seconds')
        print(f'Overall mean for {len(confirmed)} confirmed decisions: {confirmed.seconds.mean():.2f} s')
    table(decisions[['sessionId', 'runId', 'context', 'outcome', 'seconds']])


## 6. Attempts To Reach The Boss

Use a new run ID for each attempt and keep it when resuming. The selected folder represents one local recording history. Attempts are ordered across sessions within each game version. The count includes the attempt that reaches the boss. Without a recorded arrival, the result stays unavailable. Load the full history from its beginning.


In [ ]:
# An unfinished history shows observed attempts, not a completed arrival count.
attempts = pd.DataFrame(result['attempts'])
if attempts.empty:
    display(Markdown('No recorded attempts in this history.'))
else:
    table(attempts)
    labels = attempts.gameVersion
    values = [row.attemptsToBoss if row.reachedBoss else row.attemptsObserved
              for row in attempts.itertuples()]
    colors = [TEAL if reached else AMBER for reached in attempts.reachedBoss]
    fig, ax = plt.subplots()
    bars = ax.bar(labels, values, color=colors)
    ax.set_ylim(0, max(values, default=1) + 1)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    for bar, reached in zip(bars, attempts.reachedBoss):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.08,
                'Boss reached' if reached else 'Not reached yet', ha='center', fontsize=9)
    finish(ax, 'First Boss Arrival', 'Attempts (observed if boss not reached)')


## 7. Signal Activity

Signals are identified by `eventKind = signal`. The table counts occurrences by version, session and event name; the chart shows activity over session time. These records describe notifications, not numeric measurements. Older files without eventKind remain unclassified.


In [ ]:
# Group notifications by their recorded identity, without guessing from empty payloads.
signals = pd.DataFrame(result['signals'])
signal_summary = pd.DataFrame(result['signal_summary'])
if signals.empty:
    display(Markdown('No explicitly recorded Signals in this selection.'))
else:
    table(signal_summary)
    fig, ax = plt.subplots()
    for (session, event), group in signals.groupby(['sessionId', 'eventName']):
        ax.scatter(group.elapsedSeconds, [event + ' / ' + session] * len(group), label=event)
    ax.set_title('Signal Activity by Session', loc='left')
    ax.set_xlabel('Session Time (Seconds)')
    plt.tight_layout()
    plt.show()


## 8. Export And Interpretation

CSV export includes events, rooms, combats, progression, decisions, decision summaries, attempts, Signals and Signal summaries, plus warnings. The original JSONL files are preserved. Export uses the full input folder; `GAME_VERSION` filters only this notebook's views.

Use the results as measurable evidence for evaluation. Consider the game version, sample size, decision context and completeness of the recordings when drawing conclusions.


In [ ]:
# Export is opt-in and reads the full folder, independently of the view filter.
if EXPORT:
    from export_metrics import export
    count, warnings = export(DATA_DIR, OUTPUT_DIR)
    print(f'{count} records exported to {OUTPUT_DIR}; {len(warnings)} warnings.')
else:
    print('Export disabled. Set EXPORT = True to generate CSV files.')
